In [ ]:
import csv
import json

with open("/content/appoinment.csv", "r", newline="") as file:

    reader = csv.DictReader(file)

    print("Headers:", reader.fieldnames)

    rows = list(reader)

    print("Total rows:", len(rows))

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [ ]:
def parse_row(row):

    row = row.copy()

    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

In [ ]:
accepted = []
rejected = []

for row in rows:

    try:
        converted = parse_row(row)
        accepted.append(converted)

    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

Accepted: 988
Rejected: 32


**Task 1.2 — Inspect and clean, pass 1**


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/appoinment.csv")

In [ ]:
print(df)

     appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0             A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1             A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2             A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3             A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
4             A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   
...             ...        ...       ...         ...         ...   ...    ...   
1015          A0996       P415       C05  2025-01-09  2025-01-31  20.0      M   
1016          A0997       P247       C05  2025-02-23  2025-02-27  77.0      M   
1017          A0998       P422       C06  2025-05-05  2025-05-16  58.0      M   
1018          A0999       P509       C05  2025-03-30  2025-04-12  59.0      F   
1019          A1000       P085       C08  2024-12-03  2025-01-17  38.0      M   

      distance_km  fee_inr 

In [ ]:

# 1. Number of rows and columns
print(df.shape)

# 2. Information about columns
df.info()

# 3. Missing values in each column
print(df.isnull().sum())

# 4. Count duplicate rows
print(df.duplicated().sum())


(1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on      

In [ ]:

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 1000


In [ ]:

df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print(df["booking_channel"].unique())


['Phone' 'Walk-In' 'App']


In [ ]:

df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)


In [ ]:

df = df.dropna(subset=["appt_date", "status"])

print("Rows after removing missing dates/status:", len(df))


Rows after removing missing dates/status: 990


**Task 1.3: Clean, Pass 2 — Validity, Missing Values & Outliers**

In [ ]:
import numpy as np

In [ ]:
# Count invalid ages
invalid_age_count = ((df["age"] < 1) | (df["age"] > 100)).sum()

print("Invalid ages:", invalid_age_count)

# Replace invalid ages with missing value
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

Invalid ages: 5


In [ ]:
# Find median values
age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Age median:", age_median)
print("Distance median:", distance_median)

# Fill missing values
df["age"] = df["age"].fillna(age_median).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(distance_median)

Age median: 48.0
Distance median: 8.25


In [ ]:
# Calculate quartiles
Q1 = df["distance_km"].quantile(0.25)
Q3 = df["distance_km"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Count values outside limits
distance_capped = ((df["distance_km"] < lower) |
                   (df["distance_km"] > upper)).sum()

# Cap outliers
df["distance_km"] = np.clip(df["distance_km"], lower, upper)

print("Distance Q1:", Q1)
print("Distance Q3:", Q3)
print("Distance IQR:", IQR)
print("Distance lower limit:", lower)
print("Distance upper limit:", upper)
print("Distance values capped:", distance_capped)

Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Distance lower limit: -7.9375
Distance upper limit: 25.5625
Distance values capped: 26


In [ ]:
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

fee_capped = ((df["fee_inr"] < lower_fee) |
              (df["fee_inr"] > upper_fee)).sum()

df["fee_inr"] = np.clip(df["fee_inr"], lower_fee, upper_fee)

print("Fee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)
print("Fee lower limit:", lower_fee)
print("Fee upper limit:", upper_fee)
print("Fee values capped:", fee_capped)

Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Fee lower limit: 68.75
Fee upper limit: 1218.75
Fee values capped: 3


In [ ]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print(df[["age", "distance_km", "fee_inr"]].head())

   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700


**Task 1.4: Feature Engineering & NumPy Summary**

In [ ]:
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

In [ ]:
df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

In [ ]:
X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

print(X.shape)
print(X[:5])

(990, 4)
[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 [ 52.    7.5 750.    4. ]
 [ 64.   19.  700.   40. ]]


In [ ]:
column_means = np.mean(X, axis=0)

print("Column means:", column_means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:
mask = df["lead_days"] > 14

print("Mask:", mask.head())

Mask: 0    False
1    False
2    False
4    False
5     True
Name: lead_days, dtype: bool


In [ ]:
count = mask.sum()
share = mask.mean()

print("Count:", count)
print("Share:", share)
print("Percentage:", share * 100)

Count: 296
Share: 0.298989898989899
Percentage: 29.898989898989896


In [ ]:
distance = df["distance_km"].to_numpy()

z_scores = (
    distance - np.mean(distance)
) / np.std(distance)

unusual_count = (np.abs(z_scores) > 2).sum()

print("Unusual distance rows:", unusual_count)

Unusual distance rows: 53


**Task 1.5: Enrich Clinics Using a Live API**

In [ ]:
!pip install -q requests

In [ ]:
import pandas as pd
import requests

In [ ]:
clinics = pd.read_csv("/content/clinics.csv")

print(clinics.head())
print(clinics.columns)
print("Total clinics:", len(clinics))

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Index(['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude'], dtype='object')
Total clinics: 8


In [ ]:
clinic = clinics.iloc[0]

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": clinic["latitude"],
    "longitude": clinic["longitude"],
    "hourly": "temperature_2m",
    "forecast_days": 1
}

response = requests.get(
    url,
    params=params,
    timeout=10
)

print("Status code:", response.status_code)

Status code: 200


In [ ]:
if response.status_code == 200:
    data = response.json()
    print(data["hourly"]["temperature_2m"][:5])

[17.2, 16.8, 18.3, 22.1, 25.1]


In [ ]:
temp_c = data["hourly"]["temperature_2m"][0]

print("Temperature:", temp_c)

Temperature: 17.2


In [ ]:
import pandas as pd
import requests

clinics = pd.read_csv("/content/clinics.csv")

url = "https://api.open-meteo.com/v1/forecast"

temperatures = []

for index, clinic in clinics.iterrows():

    temp_c = None
    reason = None

    # Two attempts: first try + one retry
    for attempt in range(2):

        try:
            params = {
                "latitude": clinic["latitude"],
                "longitude": clinic["longitude"],
                "hourly": "temperature_2m",
                "forecast_days": 1
            }

            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            data = response.json()

            hourly = data.get("hourly")

            if not hourly or not hourly.get("temperature_2m"):
                reason = "Hourly temperature data missing"
                continue

            temp_c = hourly["temperature_2m"][0]
            reason = None
            break

        except requests.exceptions.RequestException as error:
            reason = str(error)

    if temp_c is None:
        print("Failed:", clinic["clinic_name"], "|", reason)

    temperatures.append(temp_c)

# Add temperature as a new column
clinics["temp_c"] = temperatures

# Save enriched file
clinics.to_csv("clinics_enriched.csv", index=False)

print("Total clinics:", len(clinics))
print(clinics)
print("File saved successfully!")

Total clinics: 8
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.2  
1    19.8  
2    20.8  
3    15.9  
4    15.9  
5    25.6  
6    16.9  
7    16.2  
File saved successfully!


In [ ]:
print(clinics.shape)
print(clinics.isnull().sum())
print(clinics[["clinic_name", "temp_c"]])

(8, 7)
clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
temp_c         0
dtype: int64
                clinic_name  temp_c
0     Kothrud Family Clinic    17.2
1      Hadapsar Care Centre    19.8
2    Sitabuldi Health Point    20.8
3      Malviya Nagar Clinic    15.9
4     Vaishali Nagar Clinic    15.9
5  Edappally Medical Centre    25.6
6        Vijay Nagar Clinic    16.9
7    Gomti Nagar Care Point    16.2


**Task 1.6: Export Clean Data & Generate SQL Script**

In [ ]:
# Required column order
final_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Make a copy and sort by appointment_id
final_df = df.copy()

final_df = final_df.sort_values(
    "appointment_id"
)

# Keep only the required columns, in exact order
final_df = final_df[final_columns]

print(final_df.head())
print(final_df.shape)

  appointment_id patient_id clinic_id  booked_on  appt_date  age gender  \
0          A0001       P509       C05 2025-01-24 2025-01-25   60      F   
1          A0002       P046       C07 2025-05-27 2025-06-06   39      M   
2          A0003       P009       C02 2025-03-24 2025-03-30   63      F   
4          A0004       P482       C01 2025-04-18 2025-04-22   52      F   
5          A0005       P251       C01 2024-11-23 2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
4          7.5      750             App  Consultation           Yes   
5         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  
1               2  Att

In [ ]:
# Check unique appointment IDs
print(
    "All appointment IDs unique:",
    final_df["appointment_id"].is_unique
)

# Check missing values
print(
    "Missing values:",
    final_df.isnull().sum()
)

print(
    "Total missing values:",
    final_df.isnull().sum().sum()
)

All appointment IDs unique: True
Missing values: appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Total missing values: 0


In [ ]:
final_df["booked_on"] = (
    pd.to_datetime(final_df["booked_on"])
    .dt.strftime("%Y-%m-%d")
)

final_df["appt_date"] = (
    pd.to_datetime(final_df["appt_date"])
    .dt.strftime("%Y-%m-%d")
)

# Export final CSV
final_df.to_csv(
    "appointments_clean.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


In [ ]:
check_df = pd.read_csv("appointments_clean.csv")

print("Rows and columns:", check_df.shape)
print("Unique IDs:", check_df["appointment_id"].nunique())
print("Missing values:", check_df.isnull().sum().sum())
print("Columns:", check_df.columns.tolist())

Rows and columns: (990, 16)
Unique IDs: 990
Missing values: 0
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [ ]:
import pandas as pd

def sql_value(value):

    # Missing value
    if pd.isna(value):
        return "NULL"

    # Text value
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"

    # Number value
    return str(value)

In [ ]:
print(sql_value("P196"))
print(sql_value(25))
print(sql_value(None))

'P196'
25
NULL


In [ ]:
# Read enriched clinic data
clinics_df = pd.read_csv("/content/clinics_enriched.csv")

# Exact clinic column order
clinic_columns = [
    "clinic_id",
    "clinic_name",
    "city",
    "tier",
    "latitude",
    "longitude",
    "temp_c"
]

# Create SQL file
with open("load_data.sql", "w") as file:

    # First: insert all clinics
    for row in clinics_df[clinic_columns].itertuples(
        index=False,
        name=None
    ):

        values = ", ".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO clinics VALUES ({values});\n"
        )

    # Second: insert all appointments
    for row in final_df[final_columns].itertuples(
        index=False,
        name=None
    ):

        values = ", ".join(
            sql_value(value) for value in row
        )

        file.write(
            f"INSERT INTO appointments VALUES ({values});\n"
        )

print("SQL file created successfully!")

SQL file created successfully!


In [ ]:
with open("load_data.sql", "r") as file:
    sql_lines = file.readlines()

print("Total INSERT statements:", len(sql_lines))
print("First statement:")
print(sql_lines[0])

print("Last statement:")
print(sql_lines[-1])

Total INSERT statements: 998
First statement:
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 17.2);

Last statement:
INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);

